# Frameworks de agentes de voz: del sándwich al voice-to-voice

**Lección 4 · Clase 5.3** — la lección 3 terminó con tres problemas que no eran de implementación sino de **arquitectura**: se perdía la prosodia del usuario, interrumpir era plomería difícil, y los turnos eran rígidos.

Los tres tienen la misma causa. En el sándwich, el audio se convierte en **texto** antes de que el modelo lo vea, y el texto no tiene tono. Los modelos *voice-to-voice* atacan eso por la raíz:

```
SÁNDWICH (lección 3)
  audio ─▶ STT ─▶ texto ─▶ LLM ─▶ texto ─▶ TTS ─▶ audio      3 modelos, 3 saltos

VOICE-TO-VOICE (esta lección)
  audio ────────────▶ LLM ────────────▶ audio                 1 modelo, 0 saltos
```

Un modelo como `gpt-realtime-2.1` recibe el audio y **emite audio**, en el mismo modelo y en streaming continuo. Oye que estás dudando, oye que estás molesto, y nota cuando lo interrumpes porque nunca dejó de escuchar.

Esta lección no es un notebook de ejecución: los agentes de voz necesitan micrófono, parlantes y un bucle de audio, así que viven en **scripts**. El notebook es la guía — explica el salto, mide la diferencia que importa, y compara **tres frameworks** construyendo *el mismo agente*.

## El mismo agente, cuatro veces

Para que la comparación sea entre frameworks y no entre demos, los cuatro programas resuelven el mismo caso: **Luis**, que atiende el teléfono de una ferretería y consulta el stock con una herramienta. El dominio vive en [`tienda.py`](tienda.py) y los cuatro lo importan.

| Archivo | Framework | Qué representa |
|---|---|---|
| [`nivel0_websocket_crudo.py`](nivel0_websocket_crudo.py) | ninguno | La API a pelo. Está para responder "¿qué hace el framework por mí?" |
| [`agente_openai.py`](agente_openai.py) | **OpenAI Agents SDK** | Cerrado, del mismo proveedor del modelo. Los primitivos de agentes que ya conoces. |
| [`agente_pipecat.py`](agente_pipecat.py) | **Pipecat** (Daily) | Open source. El pipeline explícito, cada pieza intercambiable. |
| [`agente_elevenlabs.py`](agente_elevenlabs.py) | **ElevenLabs Agents** | Plataforma. El agente no vive en tu código. |

In [ ]:
# Esta lección usa el entorno uv del README. NO corre en Colab: necesita micrófono local.
from dotenv import load_dotenv
import os
import subprocess
import sys

load_dotenv()

if not os.environ.get("ELEVENLABS_API_KEY") and os.environ.get("ELEVEN_API_KEY"):
    os.environ["ELEVENLABS_API_KEY"] = os.environ["ELEVEN_API_KEY"]

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_ELEVEN = bool(os.environ.get("ELEVENLABS_API_KEY"))
print("OPENAI_API_KEY presente    :", HAY_OPENAI)
print("ELEVENLABS_API_KEY presente:", HAY_ELEVEN, "(solo para agente_elevenlabs.py)")


def correr(*argumentos: str, timeout: int = 180) -> str:
    """Corre uno de los scripts de la lección y devuelve su salida."""
    proceso = subprocess.run(
        [sys.executable, *argumentos],
        capture_output=True, text=True, timeout=timeout,
        # Los scripts ya lo hacen por su cuenta; acá también, por si se corre desde el notebook.
        env={**os.environ, "NLTK_DISABLE_IMPORT_SECURITY": "1"},
    )
    salida = (proceso.stdout + proceso.stderr).strip()
    # Los logs de pipecat son ruidosos: dejamos solo lo nuestro
    return "\n".join(l for l in salida.splitlines() if not l.startswith("2026-"))

## Nivel 0: la API a pelo

Antes de comparar frameworks conviene ver qué pasa sin ninguno, porque si no, todo parece magia gratis. `nivel0_websocket_crudo.py` abre el WebSocket, manda un JSON de configuración, empuja bloques de audio en base64 y va interpretando eventos a mano.

Su modo `--smoke` hace el handshake y una pregunta de texto, sin tocar el micrófono. Sirve además como verificación de que tu llave y el modelo funcionan antes de pelearte con el audio.

In [ ]:
if HAY_OPENAI:
    print(correr("nivel0_websocket_crudo.py", "--smoke"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

Fíjate en lo que hubo que escribir a mano para eso (mira el archivo): la forma exacta del `session.update`, el decodificado de cada `delta` de audio en base64, y —el clásico— los **dos** mensajes que hay que mandar tras ejecutar una herramienta: primero el `function_call_output` y después un `response.create` para que el modelo siga hablando. Si te olvidas del segundo, el agente se queda mudo y no hay error que te lo diga.

> Un detalle que delata la edad del código que encuentres por internet: la forma del `session` **cambió**. El material original de esta clase usaba `input_audio_format: "pcm16"` y `turn_detection` en la raíz; hoy todo va anidado bajo `audio.input` / `audio.output` y el formato es un objeto (`{"type": "audio/pcm", "rate": 24000}`). Un ejemplo de 2024 no corre tal cual.

Eso —el protocolo, los formatos, el bucle de eventos, el orden de los mensajes— es exactamente lo que los tres frameworks resuelven.

## La diferencia que importa: tiempo hasta la primera sílaba

En la lección 3 medimos el sándwich completo: ~6 segundos, con el TTS aportando más de la mitad. Ahora la pregunta correcta no es cuánto demora *todo*, sino **cuánto tarda el usuario en oír la primera sílaba**, porque desde ahí ya está escuchando algo y la conversación se siente viva.

Medimos las dos arquitecturas partiendo del mismo punto —una pregunta en texto— hasta el primer byte de audio:

- **Sándwich**: el agente tiene que terminar de escribir su respuesta y solo entonces el TTS empieza a sintetizar.
- **Voice-to-voice**: el modelo empieza a emitir audio mientras todavía está decidiendo qué decir.

In [ ]:
import asyncio
import base64
import json
import time

PREGUNTA = "¿Tienen taladros? ¿Cuánto cuestan?"
MODELO_REALTIME = "gpt-realtime-2.1"


async def primer_byte_realtime() -> float:
    """Segundos desde la pregunta hasta el primer byte de audio (voice-to-voice)."""
    import websockets
    from tienda import INSTRUCCIONES

    url = f"wss://api.openai.com/v1/realtime?model={MODELO_REALTIME}"
    async with websockets.connect(
        url, additional_headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    ) as ws:
        await ws.recv()  # session.created
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                "instructions": INSTRUCCIONES,
                "output_modalities": ["audio"],
                "audio": {"output": {"format": {"type": "audio/pcm", "rate": 24000},
                                     "voice": "marin"}},
            },
        }))
        await ws.send(json.dumps({
            "type": "conversation.item.create",
            "item": {"type": "message", "role": "user",
                     "content": [{"type": "input_text", "text": PREGUNTA}]},
        }))

        inicio = time.perf_counter()
        await ws.send(json.dumps({"type": "response.create"}))
        async for crudo in ws:
            if json.loads(crudo)["type"] == "response.output_audio.delta":
                return time.perf_counter() - inicio
    return float("nan")


def primer_byte_sandwich() -> tuple[float, float]:
    """Segundos hasta el primer byte de audio con la cascada (y cuánto fue el LLM)."""
    from openai import OpenAI
    from tienda import INSTRUCCIONES

    cliente = OpenAI()
    inicio = time.perf_counter()

    respuesta = cliente.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "system", "content": INSTRUCCIONES},
                  {"role": "user", "content": PREGUNTA}],
    )
    texto = respuesta.choices[0].message.content
    tiempo_llm = time.perf_counter() - inicio

    # El TTS no puede empezar hasta tener el texto: ese es el costo de la arquitectura.
    with cliente.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts", voice="marin", input=texto, response_format="pcm",
    ) as flujo:
        for _ in flujo.iter_bytes(chunk_size=1024):
            break  # el primer trozo y listo
    return time.perf_counter() - inicio, tiempo_llm


if HAY_OPENAI:
    total_sandwich, tiempo_llm = primer_byte_sandwich()
    # `await` directo: el kernel de Jupyter ya tiene un event loop corriendo,
    # así que asyncio.run() daría RuntimeError. En un script sí se usa asyncio.run().
    tiempo_v2v = await primer_byte_realtime()

    print(f"{'arquitectura':<34} {'primera sílaba':>15}")
    print("-" * 52)
    print(f"{'Sándwich (LLM → TTS)':<34} {total_sandwich:>14.2f}s")
    print(f"{'   del cual, el LLM':<34} {tiempo_llm:>14.2f}s")
    print(f"{'Voice-to-voice (gpt-realtime-2.1)':<34} {tiempo_v2v:>14.2f}s")
    print("-" * 52)
    if tiempo_v2v < total_sandwich:
        print(f"→ el voice-to-voice habla {total_sandwich / tiempo_v2v:.1f}× antes")
else:
    print("⛔ Falta OPENAI_API_KEY.")

La diferencia no viene de que un modelo sea "más rápido": viene de que en el sándwich hay una **barrera**. El TTS no puede empezar hasta que el LLM haya terminado de escribir, porque necesita el texto. El modelo voice-to-voice no tiene esa barrera — está generando audio y contenido a la vez.

Y ojo con lo que esta medición **no** captura, que es la mitad del argumento: partimos de una pregunta en *texto*, así que le regalamos al sándwich la transcripción (que en la lección 3 costó ~1,3 s más). Con audio real, la brecha es más grande.

Tampoco captura lo que no es tiempo: que el modelo oiga tu tono, y que puedas interrumpirlo. Eso no se mide con un cronómetro, se prueba hablando — y para eso están los scripts.

## Framework 1 · OpenAI Agents SDK

El framework de agentes del mismo proveedor del modelo. Su gracia es la **continuidad**: `RealtimeAgent` usa los mismos primitivos que los agentes de texto que ya escribiste —instrucciones, `@function_tool`, handoffs, guardrails— y encima resuelve el protocolo.

```python
agente = RealtimeAgent(name="Luis", instructions=INSTRUCCIONES, tools=[consultar_stock_tool])

runner = RealtimeRunner(starting_agent=agente, config={
    "model_settings": {
        "model_name": "gpt-realtime-2.1",
        "audio": {
            "input": {"format": "pcm16",
                      "turn_detection": {"type": "semantic_vad", "interrupt_response": True}},
            "output": {"format": "pcm16", "voice": "marin"},
        },
    },
})
sesion = await runner.run()
```

Lo que **no** resuelve: el micrófono y los parlantes. Ese cableado con PyAudio es la mitad de [`agente_openai.py`](agente_openai.py) y explica por qué es el archivo más largo de los tres.

Vale detenerse en `semantic_vad`. Un VAD clásico corta el turno por **silencio**, así que si dudas a mitad de frase ("quiero… un taladro") te interrumpe. El semántico usa el contenido para decidir si terminaste. Es la clase de detalle que separa una demo de un producto, y acá es un string en la configuración.

In [ ]:
if HAY_OPENAI:
    print(correr("agente_openai.py", "--smoke"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

## Framework 2 · Pipecat (open source)

Pipecat, de Daily, modela la conversación como un **pipeline de frames**: el audio entra por un extremo, atraviesa una lista de procesadores y sale por el otro.

Ese modelo tiene una consecuencia que es el mejor argumento pedagógico de toda la lección: las dos arquitecturas de esta clase son **la misma lista, con dos elementos de diferencia**.

In [ ]:
if HAY_OPENAI:
    print(correr("agente_pipecat.py", "--check", "--modo", "cascada"))
    print()
    print(correr("agente_pipecat.py", "--check", "--modo", "realtime"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

Ahí está, literal: en el modo `realtime` desaparecen `OpenAISTTService` y `OpenAITTSService`. El servicio de LLM **es** el modelo de voz. Y no es un detalle de este script — Pipecat trae sus propias plantillas `bot_cascade` y `bot_realtime`, o sea que la distinción que hicimos entre la lección 3 y la 4 es la misma que hace el framework.

Las dos cosas que Pipecat da y los otros no:

- **Cada pieza es intercambiable.** Cambiar el STT de OpenAI a Deepgram, o el TTS a ElevenLabs o Cartesia, es cambiar una línea. Si tu apuesta es no quedar amarrado a un proveedor, esto es el argumento.
- **VAD local.** El `SileroVADAnalyzer` corre en tu máquina: la detección de silencios no manda audio a ningún servidor. Para datos sensibles, importa.

Su costo es la superficie: es el framework con más conceptos que aprender (frames, procesadores, agregadores de contexto, workers) y el que más se mueve entre versiones.

> **Una trampa concreta que te vas a encontrar.** Pipecat importa `nltk`, que trae un hook de seguridad que bloquea cualquier módulo cuyo origen esté *dentro del directorio de trabajo*. Como nuestro `.venv` vive justamente ahí —la convención de `uv`— el import se cae con un mensaje bastante desorientador sobre `regex`. Se arregla con `NLTK_DISABLE_IMPORT_SECURITY=1`, y los scripts de esta lección ya lo hacen por su cuenta.

## Framework 3 · ElevenLabs Agents (plataforma)

Acá el cambio es de fondo, no de sintaxis: **el agente no vive en tu código**. Se crea como un recurso en la plataforma —prompt, idioma, voz, modelo, herramientas— y tu programa solo abre una conversación contra ese recurso y le presta el micrófono.

```python
creado = cliente.conversational_ai.agents.create(name="Luis", conversation_config={...})

conversacion = Conversation(
    cliente, creado.agent_id, requires_auth=True,
    audio_interface=DefaultAudioInterface(),   # micrófono y parlantes, resueltos
    client_tools=herramientas,
)
conversacion.start_session()
```

Lo que ganas es mucho: cero plomería de audio, turnos e interrupciones resueltos, telefonía incluida, y un panel con las conversaciones grabadas y transcritas — observabilidad que en los otros dos tienes que construir. El prompt se edita desde el dashboard sin volver a desplegar.

Lo que entregas también es mucho: la lógica conversacional queda en un proveedor. Migrarla después no es cambiar una línea, es un proyecto.

Dos detalles prácticos que descubrimos preparando la lección: un agente en español **debe** usar `eleven_flash_v2_5` o `turbo` (la API rechaza los demás modelos), y hay que usar una voz *premade*, porque las de la biblioteca —incluidas las de acento chileno— requieren plan pago.

El script crea el agente, conversa y lo **borra al salir**, para no dejar basura en la cuenta.

In [ ]:
if HAY_ELEVEN:
    print(correr("agente_elevenlabs.py", "--check"))
else:
    print("⚠️ Falta ELEVENLABS_API_KEY — se salta (llave gratis en elevenlabs.io).")

## La tabla de decisión

|  | Nivel 0 (crudo) | OpenAI Agents SDK | Pipecat | ElevenLabs Agents |
|---|---|---|---|---|
| **Licencia** | — | cerrado | **open source** (BSD) | cerrado |
| **Dónde vive la lógica** | tu código | tu código | tu código | **la plataforma** |
| **Audio local resuelto** | no | no | **sí** | **sí** |
| **Turnos / interrupciones** | tú | el servidor (`semantic_vad`) | VAD local + servidor | la plataforma |
| **Cambiar de proveedor** | reescribir | atado a OpenAI | **una línea** | no aplica |
| **Telefonía / SIP** | tú | vía API | sí (Twilio, Daily) | **incluida** |
| **Observabilidad** | tú | traces del SDK | métricas + observers | **panel con grabaciones** |
| **Autohospedable** | sí (salvo el modelo) | no | **sí** | no |
| **Curva de aprendizaje** | alta | baja | media-alta | **muy baja** |
| **Líneas para este agente** | ~200 | ~180 | ~200 (dos arquitecturas) | ~140 |

### Cómo elegir, en una frase cada uno

- **ElevenLabs Agents** si el agente de voz *es* el producto y quieres estar en producción esta semana. Es el camino más corto a algo que funciona bien, y el más caro de abandonar.
- **Pipecat** si te importa no quedar amarrado, si necesitas mezclar proveedores, si tienes requisitos de datos que empujan a autohospedar, o si el agente de voz es una pieza de un sistema más grande que tú controlas.
- **OpenAI Agents SDK** si ya vives en OpenAI y tu agente de voz es la extensión de agentes de texto que ya tienes. La continuidad de primitivos vale mucho más de lo que parece.
- **Nivel 0** casi nunca en producción — pero léelo una vez, porque cuando algo falle en cualquiera de los tres, vas a estar depurando esto.

> Y el que no está en la tabla: **el sándwich de la lección 3**. Sigue siendo la respuesta correcta cuando la conversación no es en tiempo real (procesar grabaciones, un buzón de voz, un flujo asíncrono), cuando necesitas el texto como artefacto auditable de cada turno, o cuando quieres un modelo de razonamiento en el medio y puedes pagar la espera. Voice-to-voice no reemplaza al sándwich: resuelve el caso conversacional.

## Ahora háblales

Nada de lo anterior reemplaza la prueba real. Los cuatro scripts se corren desde la terminal, en esta carpeta, y se salen con Ctrl-C:

```bash
uv run python agente_openai.py                    # OpenAI Agents SDK
uv run python agente_pipecat.py                   # Pipecat, voice-to-voice
uv run python agente_pipecat.py --modo cascada    # Pipecat, el sándwich — compara al oído
uv run python agente_elevenlabs.py                # ElevenLabs Agents
uv run python nivel0_websocket_crudo.py           # sin framework
```

Tres cosas que vale la pena probar a propósito, porque son justo lo que el sándwich no podía:

1. **Interrúmpelo.** Empieza a hablar mientras Luis responde. Debería callarse en el acto.
2. **Duda a mitad de frase.** "Quiero… un… taladro." Un VAD por silencio te habría cortado; el semántico espera.
3. **Cambia el tono, no las palabras.** Pregunta lo mismo apurado y después relajado, o molesto. En el sándwich el agente veía el mismo texto en los dos casos. Acá no.

Y compara al oído `--modo cascada` contra `--modo realtime` de Pipecat: es el mismo agente, la misma herramienta y el mismo código, con dos elementos de diferencia en una lista.

## Qué nos llevamos

- El salto de sándwich a **voice-to-voice** no es una optimización, es un cambio de arquitectura: se elimina la barrera de "esperar el texto completo" y se deja de tirar a la basura el tono del usuario.
- La métrica correcta en voz no es el tiempo total sino el **tiempo hasta la primera sílaba**, y ahí la diferencia que medimos es de varias veces — con el agravante de que la medición le regaló al sándwich la transcripción.
- Los frameworks no compiten en features sino en **dónde ponen la frontera**: Pipecat te da el pipeline completo y la libertad de proveedor; el SDK de OpenAI te da continuidad con tus agentes de texto; ElevenLabs te da el producto casi armado a cambio de que la lógica viva allá.
- Ninguno resuelve todo: el SDK de OpenAI no toca tu micrófono, Pipecat te cobra en conceptos, ElevenLabs te cobra en dependencia.
- Y el sándwich **no murió**: sigue ganando en lo asíncrono, en lo auditable y cuando necesitas razonamiento en el medio.

Con esto cierra la clase 5.3: la señal (L1), la síntesis (L2), la arquitectura en cascada (L3) y los modelos que oyen y hablan directo (L4).